<a href="https://colab.research.google.com/github/YaminisriVellanki/24B11DS221_DriveReady_OPC_2026/blob/main/task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 – Agent Memory and RAG

Today, we are enhancing our Day 4 AI Agent by adding memory and Retrieval-Augmented Generation (RAG). The agent will remember conversations, retrieve relevant business information from a knowledge base, and provide more accurate answers to customers.

In [8]:
# Install packages required for Day 5
!pip install -q -U langgraph langchain langchain-core langchain-groq chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [10]:
import importlib.metadata

import langgraph
import langchain
import chromadb

from langgraph.checkpoint.memory import InMemorySaver

print("LangGraph:", importlib.metadata.version("langgraph"))
print("LangChain:", importlib.metadata.version("langchain"))
print("ChromaDB:", chromadb.__version__)

print("\nAll required packages imported successfully!")

LangGraph: 1.2.24
LangChain: 1.5.2
ChromaDB: 1.5.9

All required packages imported successfully!


## Task 1: Understanding Memory in AI Agents

AI agents use memory to retain information and provide better responses.

There are three types of memory:

1. Conversation Memory – remembers the ongoing conversation.
2. Customer Memory – stores customer-specific information and preferences.
3. Knowledge Memory – stores business information, policies, FAQs, and reference data.

In Sweet Crumbs Home Bakery:

- Conversation memory remembers what the customer said earlier.
- Customer memory stores details about a particular customer.
- Knowledge memory stores bakery information that the agent can retrieve when needed.

In [11]:
memory_types = {
    "Conversation Memory": "Remembers the ongoing conversation and previous messages.",
    "Customer Memory": "Stores customer-specific information and preferences.",
    "Knowledge Memory": "Stores business information, FAQs, and policies."
}

for memory, description in memory_types.items():
    print(f"\n{memory}")
    print(description)


Conversation Memory
Remembers the ongoing conversation and previous messages.

Customer Memory
Stores customer-specific information and preferences.

Knowledge Memory
Stores business information, FAQs, and policies.


## Task 2: Conversation Memory using Checkpointer

Conversation memory allows an AI agent to remember previous messages
within a conversation.

A checkpointer saves the conversation state.

A Thread ID identifies a particular conversation. When the same Thread ID
is reused, the agent can access its previous conversation history.

Different Thread IDs maintain separate conversation histories.

We use InMemorySaver to store conversation state during execution.

In [12]:
from langgraph.checkpoint.memory import InMemorySaver

# Create a conversation memory checkpointer
checkpointer = InMemorySaver()

print("Conversation checkpointer created successfully.")

Conversation checkpointer created successfully.


## Task 3: Conversation Threads

A Thread ID uniquely identifies a conversation.

When the same Thread ID is used again, the agent can continue its
previous conversation using the stored state.

Different Thread IDs maintain separate conversation histories.

We use two customers to demonstrate how conversation threads are isolated.

In [13]:
# Simulate two separate customer conversation threads

customer_threads = {
    "Customer_1": "thread_customer_1",
    "Customer_2": "thread_customer_2"
}

for customer, thread_id in customer_threads.items():
    print(f"{customer} -> {thread_id}")

Customer_1 -> thread_customer_1
Customer_2 -> thread_customer_2


## Task 4: Prompt Efficiency and Context Window

An AI model receives information through a prompt.

The Context Window is the maximum amount of information (tokens) that
a model can process at one time.

If we send the entire conversation, business documents, FAQs, and
customer history every time:

- The prompt becomes unnecessarily large.
- Token consumption increases.
- Response time may increase.
- Relevant information can become difficult to focus on.

Instead, an AI agent should retrieve only the information required
for the current question.

Memory and RAG help the agent access relevant information without
repeatedly sending everything.

In [14]:
# Simulate the difference between sending all information and relevant information

full_context = """
Bakery menu, prices, delivery policies, payment policies, customer history,
all previous conversations, cake ingredients, cancellation policies,
business hours, birthday orders, wedding orders, product descriptions.
""" * 30

relevant_context = """
Plum cake price: Rs. 500
"""

print("Full context characters:", len(full_context))
print("Relevant context characters:", len(relevant_context))
print("\nFull context consumes more input.")
print("Retrieving only relevant information makes prompts more efficient.")

Full context characters: 6510
Relevant context characters: 26

Full context consumes more input.
Retrieving only relevant information makes prompts more efficient.


## Task 5: Understanding Embeddings

Embeddings convert text into numerical vectors that represent its meaning.

Texts with similar meanings have vectors that are closer together
in vector space.

This allows AI agents to retrieve information based on meaning
rather than only exact keyword matches.

Example:

"How much does a plum cake cost?"

"Tell me the price of plum cake."

Both questions have similar meanings, even though the wording differs.

Embeddings help us perform semantic search, which is useful for RAG.

In [15]:
texts = [
    "How much does a plum cake cost?",
    "Tell me the price of plum cake.",
    "What are your bakery opening hours?",
    "How long is the bakery open?"
]

for i, text in enumerate(texts, start=1):
    print(f"{i}. {text}")

1. How much does a plum cake cost?
2. Tell me the price of plum cake.
3. What are your bakery opening hours?
4. How long is the bakery open?


## Task 6: ChromaDB Vector Database

ChromaDB is a vector database used to store and retrieve information
based on semantic similarity.

It stores documents along with their vector representations.

When a user asks a question, the question is converted into a vector
and compared with stored vectors to retrieve relevant information.

In our Sweet Crumbs Home Bakery agent, ChromaDB will store bakery
FAQs, product information, and business policies.

In [16]:
import chromadb

# Create ChromaDB client
chroma_client = chromadb.Client()

# Create or retrieve a collection
collection = chroma_client.get_or_create_collection(
    name="sweet_crumbs_knowledge"
)

print("ChromaDB initialized successfully!")
print("Collection:", collection.name)

ChromaDB initialized successfully!
Collection: sweet_crumbs_knowledge


## Task 7: Storing Knowledge in ChromaDB

A vector database stores information in a way that allows semantic retrieval.

We add documents containing bakery information to our ChromaDB collection.

Each document is assigned a unique ID.

This stored knowledge will be used later by our RAG system to answer
customer queries based on relevant information.

In [17]:
bakery_facts = [
    "Sweet Crumbs Home Bakery sells plum cake for Rs. 500.",
    "Sweet Crumbs Home Bakery sells chocolate cake for Rs. 650.",
    "Sweet Crumbs Home Bakery sells cupcakes for Rs. 80.",
    "Customers can pay using UPI or cash.",
    "Customers should place regular cake orders at least 24 hours in advance.",
    "Custom photo cakes require at least 48 hours of advance notice.",
    "Free candles are provided with cake orders.",
    "Customers can include a cake message of up to 20 characters.",
    "Plum cake contains nuts.",
    "Cupcakes are nut-free.",
    "Cakes should be refrigerated when not being consumed.",
    "Cakes are best consumed within 3 days.",
    "Delivery is available within 5 km.",
    "Delivery charges are Rs. 50.",
    "Delivery is free for orders above Rs. 1000.",
    "Orders above Rs. 1000 require a 50 percent advance payment.",
    "Cancellation is free if requested 12 hours before delivery.",
]

collection.add(
    ids=[f"bakery_fact_{i}" for i in range(len(bakery_facts))],
    documents=bakery_facts
)

print("Stored", collection.count(), "documents in ChromaDB.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 47.3MiB/s]


Stored 17 documents in ChromaDB.


## Task 8: Semantic Retrieval from ChromaDB

ChromaDB allows us to retrieve relevant documents using vector similarity.

When a customer asks a question:
1. The question is processed.
2. ChromaDB searches for semantically similar documents.
3. The most relevant documents are returned.

This allows the agent to retrieve relevant knowledge even when
the customer's wording differs from the stored documents.

In [20]:
# Search the knowledge base using a customer question

question = "How early should I book a customized cake?"

results = collection.query(
    query_texts=[question],
    n_results=3
)

print("Customer question:", question)
print("\nRetrieved information:\n")

for document in results["documents"][0]:
    print("-", document)

Customer question: How early should I book a customized cake?

Retrieved information:

- Customers should place regular cake orders at least 24 hours in advance.
- Custom photo cakes require at least 48 hours of advance notice.
- Cakes are best consumed within 3 days.


## Task 9: Chunking

Chunking is the process of dividing large documents into smaller pieces
of text called chunks.

Why chunking is important:
- Large documents are difficult to retrieve efficiently.
- Smaller chunks provide more focused information.
- Relevant information can be retrieved without loading the entire document.
- Chunk size and overlap affect retrieval quality.

Chunk Overlap is the repeated text shared between consecutive chunks.
It helps preserve context when a document is split.

In [21]:
# Sample bakery policy document
policy_document = """
Sweet Crumbs Home Bakery accepts cake orders for birthdays and celebrations.
Regular cakes must be ordered at least 24 hours in advance.
Custom photo cakes require 48 hours advance notice.
Free candles are provided with cake orders.
Customers can include a cake message of up to 20 characters.
Cakes should be refrigerated when not being consumed.
Cakes are best consumed within 3 days.
"""

def split_into_chunks(text, chunk_size=40, overlap=8):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = words[start:end]
        chunks.append(" ".join(chunk))

        if end >= len(words):
            break

        start = end - overlap

    return chunks

chunks = split_into_chunks(
    policy_document,
    chunk_size=40,
    overlap=8
)

for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}:\n{chunk}")


Chunk 1:
Sweet Crumbs Home Bakery accepts cake orders for birthdays and celebrations. Regular cakes must be ordered at least 24 hours in advance. Custom photo cakes require 48 hours advance notice. Free candles are provided with cake orders. Customers can include

Chunk 2:
are provided with cake orders. Customers can include a cake message of up to 20 characters. Cakes should be refrigerated when not being consumed. Cakes are best consumed within 3 days.


## Task 10: Creating the search_faq Tool

A retrieval tool allows an AI agent to search its knowledge base
whenever it needs additional information.

Our search_faq tool:

1. Receives a customer question.
2. Searches ChromaDB for relevant documents.
3. Retrieves the most relevant facts.
4. Returns the information to the agent.

The agent uses these retrieved facts to answer questions accurately.

If relevant information is unavailable, the agent should not invent an answer.

In [22]:
from langchain.tools import tool

@tool
def search_faq(question: str) -> str:
    """Search Sweet Crumbs Home Bakery knowledge base for answers about products, prices, policies, delivery, ingredients, and orders."""

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    documents = results.get("documents", [[]])[0]

    if not documents:
        return "No relevant information was found in the bakery knowledge base."

    return "\n".join(
        f"- {document}" for document in documents
    )

print("RAG retrieval tool created:", search_faq.name)
print(search_faq.description)

RAG retrieval tool created: search_faq
Search Sweet Crumbs Home Bakery knowledge base for answers about products, prices, policies, delivery, ingredients, and orders.


In [23]:
question = "Do you provide free delivery for large orders?"

response = search_faq.invoke({"question": question})

print("Question:", question)
print("\nRetrieved knowledge:")
print(response)

Question: Do you provide free delivery for large orders?

Retrieved knowledge:
- Delivery is free for orders above Rs. 1000.
- Delivery charges are Rs. 50.
- Delivery is available within 5 km.


## Task 11: RAG Integration with the AI Agent

Retrieval-Augmented Generation (RAG) combines information retrieval
with AI text generation.

RAG works in three stages:

1. Retrieval – relevant information is retrieved from ChromaDB.
2. Augmentation – retrieved information is added to the agent's context.
3. Generation – the AI generates an answer grounded in the retrieved information.

Our Sweet Crumbs Home Bakery agent uses the search_faq tool to
retrieve bakery information whenever necessary.

The agent should not invent bakery policies when information
is unavailable.

In [74]:
!pip install -q -U langchain langchain-groq langchain-core langgraph chromadb gspread

In [75]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ").strip()

Enter Groq API Key: ··········


In [76]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

print("LLM initialized successfully!")

LLM initialized successfully!


In [77]:
from pydantic import BaseModel, Field
from typing import Optional

class BakeryOrder(BaseModel):
    customer_name: Optional[str] = Field(default=None, description="Name of the customer")
    item: Optional[str] = Field(default=None, description="Bakery item requested")
    quantity: Optional[int] = Field(default=None, description="Quantity requested")
    order_date: Optional[str] = Field(default=None, description="Requested date for the order")
    eggless: Optional[bool] = Field(default=None, description="Whether an eggless item is requested")

structured_llm = llm.with_structured_output(BakeryOrder)

print("Order schema ready!")

Order schema ready!


In [78]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

credentials, _ = default()
gc = gspread.authorize(credentials)

SHEET_NAME = "Sweet Crumbs Bakery Orders"

try:
    spreadsheet = gc.open(SHEET_NAME)
except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)

worksheet = spreadsheet.sheet1

if not worksheet.get_all_values():
    worksheet.append_row([
        "Customer Name",
        "Item",
        "Quantity",
        "Order Date",
        "Eggless"
    ])

print("Google Sheets connected!")
print("Spreadsheet URL:", spreadsheet.url)

Google Sheets connected!
Spreadsheet URL: https://docs.google.com/spreadsheets/d/1frptxvlWsUGpuC8Q-ZXpBqPoHK82WCPnjExb25G1plw


In [79]:
from langchain.tools import tool

@tool
def get_bakery_price(item: str) -> str:
    """Find the price of a bakery item from the menu."""

    menu = {
        "plum cake": 500,
        "chocolate cake": 650,
        "cupcake": 80
    }

    item = item.lower().strip()

    if item in menu:
        return f"{item} costs Rs. {menu[item]}."

    return "The requested item is not available in the menu."

In [80]:
@tool
def check_order_requirement() -> str:
    """Check the advance notice requirement for bakery orders."""

    return "Orders require at least 24 hours advance notice."

In [81]:
@tool
def log_bakery_order(
    customer_name: str,
    item: str,
    quantity: int,
    order_date: str,
    eggless: bool
) -> str:
    """Save a customer bakery order to the Google Sheets order register."""

    worksheet.append_row([
        customer_name,
        item,
        quantity,
        order_date,
        eggless
    ])

    return "Order saved successfully."

In [82]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="sweet_crumbs_knowledge"
)

print("ChromaDB ready!")
print("Stored facts:", collection.count())

ChromaDB ready!
Stored facts: 17


In [83]:
@tool
def search_faq(question: str) -> str:
    """Search Sweet Crumbs Home Bakery knowledge base for answers about products, prices, policies, delivery, ingredients, and orders."""

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    documents = results.get("documents", [[]])[0]

    if not documents:
        return "No relevant information was found in the bakery knowledge base."

    return "\n".join(f"- {document}" for document in documents)

In [84]:
SYSTEM_PROMPT = """
You are the AI assistant for Sweet Crumbs Home Bakery.

Your responsibilities:
1. Answer customer questions politely and clearly.
2. Use get_bakery_price for menu price questions.
3. Use check_order_requirement for order advance notice.
4. Use log_bakery_order only when the customer confirms an order
   and all required details are available.
5. Use search_faq to retrieve bakery policies and knowledge
   stored in the knowledge base.
6. Do not invent prices, policies, ingredients, or business facts.
7. If the knowledge base does not contain the requested information,
   clearly say that the information is unavailable.
8. Remember previous messages within the same conversation.
9. Never mix information from different customers' threads.
"""

In [85]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Fresh conversation memory
checkpointer = InMemorySaver()

# Day 4 tools + Day 5 RAG tool
day5_tools = [
    get_bakery_price,
    check_order_requirement,
    log_bakery_order,
    search_faq
]

SYSTEM_PROMPT = """
You are the AI assistant for Sweet Crumbs Home Bakery.

Rules:
1. Answer customers politely and clearly.
2. Use get_bakery_price for menu price questions.
3. Use check_order_requirement for order advance notice.
4. Use log_bakery_order only when the customer explicitly confirms
   an order and all required details are available.
5. Use search_faq for bakery information and policies stored in ChromaDB.
6. Never invent prices, ingredients, policies, or business facts.
7. If information is unavailable, clearly say you do not have that information.
8. Remember messages within the same conversation thread.
9. Never mix information between different customer threads.
10. Ask for missing details before placing an order.
"""

bakery_agent = create_agent(
    model=llm,
    tools=day5_tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

print("Day 5 Bakery Agent created successfully!")
print("Tools:", [tool.name for tool in day5_tools])
print("Conversation memory enabled!")
print("RAG integration enabled!")

Day 5 Bakery Agent created successfully!
Tools: ['get_bakery_price', 'check_order_requirement', 'log_bakery_order', 'search_faq']
Conversation memory enabled!
RAG integration enabled!


In [86]:
config = {
    "configurable": {
        "thread_id": "bakery_test_price"
    }
}

response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the price of a chocolate cake?"}
        ]
    },
    config=config
)

print(response["messages"][-1].content)

The chocolate cake is priced at **Rs. 650**. Let me know if you’d like any more details or would like to place an order!


In [87]:
response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Do you provide free delivery for large orders?"}
        ]
    },
    config={
        "configurable": {
            "thread_id": "bakery_test_rag"
        }
    }
)

print(response["messages"][-1].content)

Yes! We offer free delivery for any order that exceeds **Rs 1000**. Orders below that amount incur a delivery charge of Rs 50. Let me know if you’d like help with anything else—perhaps checking the total for a specific order or placing one.


In [88]:
memory_config = {
    "configurable": {
        "thread_id": "bakery_test_memory"
    }
}

# First message
bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "I am Ravi. I am interested in plum cake."}
        ]
    },
    config=memory_config
)

# Second message — agent should remember
response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Which cake was I interested in?"}
        ]
    },
    config=memory_config
)

print(response["messages"][-1].content)

You mentioned that you’re interested in our **plum cake**.


In [89]:
response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "How long in advance should I order a cake?"}
        ]
    },
    config={
        "configurable": {
            "thread_id": "bakery_test_policy"
        }
    }
)

print(response["messages"][-1].content)

We kindly ask that you place cake orders at least **24 hours** in advance so we have enough time to bake and decorate it to perfection. Let us know if you’d like to schedule a cake, and I’ll be happy to help with the details!


### Task 12: Testing Customer Memory Isolation

Each customer is assigned a unique thread ID. Conversation memory is maintained separately for each thread, allowing the agent to remember individual customer preferences without mixing information between customers.

In this task, two customers provide different cake preferences. The agent is then tested to verify that it recalls the correct preference for each customer.

In [90]:
customer_A = {
    "configurable": {
        "thread_id": "customer_A"
    }
}

customer_B = {
    "configurable": {
        "thread_id": "customer_B"
    }
}

In [91]:
# Customer A
bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "My name is Ravi. I like plum cake."}
        ]
    },
    config=customer_A
)

# Customer B
bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "My name is Anu. I like chocolate cake."}
        ]
    },
    config=customer_B
)

print("Both customer conversations stored.")

Both customer conversations stored.


In [92]:
# Ask Customer A
response_A = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What cake do I like?"}
        ]
    },
    config=customer_A
)

# Ask Customer B
response_B = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What cake do I like?"}
        ]
    },
    config=customer_B
)

print("Customer A:", response_A["messages"][-1].content)
print("Customer B:", response_B["messages"][-1].content)

Customer A: You mentioned that you like plum cake. 🍰
Customer B: You mentioned that you like chocolate cake. 🍫🎂 Let me know if you’d like more details about our chocolate cake options, pricing, or if you’d like to place an order!


### Conclusion

Customer memory isolation was tested using separate thread IDs. The agent maintained separate conversation histories for both customers and recalled their preferences correctly. This demonstrates thread-based memory isolation and helps prevent customer information from being mixed between conversations.

### Task 13: Handling Unsupported Questions

The RAG system retrieves information from a ChromaDB knowledge base. However, vector databases can return the closest matching documents even when a query is unrelated to the stored knowledge.

To prevent incorrect answers, a relevance threshold is introduced. If the retrieved documents are not sufficiently relevant, the system returns a message indicating that the requested information is unavailable.

This improves reliability and reduces hallucinations by grounding responses in the bakery's stored knowledge.

In [93]:
test_questions = [
    "What is the price of plum cake?",
    "Do you offer free delivery?",
    "What is the bakery Wi-Fi password?",
    "How many chairs are available in the bakery?"
]

for question in test_questions:
    results = collection.query(
        query_texts=[question],
        n_results=3,
        include=["documents", "distances"]
    )

    print("\nQuestion:", question)
    for document, distance in zip(
        results["documents"][0],
        results["distances"][0]
    ):
        print(f"Distance: {distance:.4f} | {document}")


Question: What is the price of plum cake?
Distance: 0.5766 | Sweet Crumbs Home Bakery sells plum cake for Rs. 500.
Distance: 0.6895 | Plum cake contains nuts.
Distance: 0.9251 | Sweet Crumbs Home Bakery sells chocolate cake for Rs. 650.

Question: Do you offer free delivery?
Distance: 0.6024 | Delivery is free for orders above Rs. 1000.
Distance: 0.8532 | Delivery charges are Rs. 50.
Distance: 0.9758 | Cancellation is free if requested 12 hours before delivery.

Question: What is the bakery Wi-Fi password?
Distance: 1.3823 | Sweet Crumbs Home Bakery sells chocolate cake for Rs. 650.
Distance: 1.4216 | Sweet Crumbs Home Bakery sells cupcakes for Rs. 80.
Distance: 1.4276 | Sweet Crumbs Home Bakery sells plum cake for Rs. 500.

Question: How many chairs are available in the bakery?
Distance: 1.0911 | Sweet Crumbs Home Bakery sells chocolate cake for Rs. 650.
Distance: 1.1355 | Customers can include a cake message of up to 20 characters.
Distance: 1.2071 | Sweet Crumbs Home Bakery sells p

In [94]:
from langchain.tools import tool

RELEVANCE_THRESHOLD = 1.0

@tool
def search_faq(question: str) -> str:
    """Search the bakery knowledge base and return only relevant information."""

    results = collection.query(
        query_texts=[question],
        n_results=3,
        include=["documents", "distances"]
    )

    documents = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]

    if not documents or not distances:
        return "No relevant information was found in the bakery knowledge base."

    relevant_documents = [
        document
        for document, distance in zip(documents, distances)
        if distance <= RELEVANCE_THRESHOLD
    ]

    if not relevant_documents:
        return "The requested information is not available in the bakery knowledge base."

    return "\n".join(
        f"- {document}" for document in relevant_documents
    )

print("Updated RAG retrieval tool created.")

Updated RAG retrieval tool created.


In [95]:
bakery_agent = create_agent(
    model=llm,
    tools=[
        get_bakery_price,
        check_order_requirement,
        log_bakery_order,
        search_faq
    ],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

print("Agent updated with relevance-checked RAG!")

Agent updated with relevance-checked RAG!


In [96]:
unsupported_questions = [
    "What is the Wi-Fi password of the bakery?",
    "How many chairs are available inside the bakery?",
    "Do you have a television inside the bakery?"
]

for question in unsupported_questions:
    response = bakery_agent.invoke(
        {
            "messages": [
                {"role": "user", "content": question}
            ]
        },
        config={
            "configurable": {
                "thread_id": f"unsupported_{unsupported_questions.index(question)}"
            }
        }
    )

    print("\nQuestion:", question)
    print("Answer:", response["messages"][-1].content)
    print("-" * 60)


Question: What is the Wi-Fi password of the bakery?
Answer: I’m sorry, but I don’t have the Wi‑Fi password information available. If there’s anything else I can help you with—such as our menu, ordering details, or bakery policies—please let me know!
------------------------------------------------------------

Question: How many chairs are available inside the bakery?
Answer: I’m sorry, but I don’t have information about the number of chairs available inside the bakery. If you have any other questions about our baked goods, ordering, or policies, I’d be happy to help!
------------------------------------------------------------

Question: Do you have a television inside the bakery?
Answer: I’m sorry, but I don’t have information about whether there’s a television inside the bakery. If you have any other questions about our baked goods or services, I’d be happy to help!
------------------------------------------------------------


### Conclusion

A relevance threshold was introduced into the RAG retrieval system to filter out unrelated information. Unsupported questions were tested to ensure the agent does not generate answers without knowledge base evidence.

This improves the reliability, accuracy, and trustworthiness of the bakery AI agent.

### Task 14: Agent Sign-off Testing

The integrated AI bakery agent is tested against practical customer scenarios covering product pricing, business policies, order creation, conversation memory, order modification, and unsupported questions.

Each test evaluates whether the agent selects the appropriate tool, retrieves relevant knowledge, maintains conversation context, and responds accurately without hallucinating information.

In [97]:
test_config = {
    "configurable": {
        "thread_id": "signoff_price_test"
    }
}

response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "How much does a chocolate cake cost?"}
        ]
    },
    config=test_config
)

print("Agent:", response["messages"][-1].content)

Agent: The chocolate cake is priced at **Rs. 650**. Let me know if you’d like any more details or would like to place an order!


In [98]:
response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What payment methods do you accept?"}
        ]
    },
    config={"configurable": {"thread_id": "signoff_payment_test"}}
)

print("Agent:", response["messages"][-1].content)

Agent: I’m sorry, but I don’t have information about the payment methods we accept. If you have any other questions about our bakery items or policies, feel free to ask!


In [99]:
response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "How long can I keep the cake fresh?"}
        ]
    },
    config={"configurable": {"thread_id": "signoff_freshness_test"}}
)

print("Agent:", response["messages"][-1].content)

Agent: Cakes stay fresh for up to **3 days** when stored properly. To keep them at their best, place the cake in an airtight container and refrigerate it when you’re not serving it. Enjoy!


In [100]:
response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is your bakery Wi-Fi password?"}
        ]
    },
    config={"configurable": {"thread_id": "signoff_unsupported_test"}}
)

print("Agent:", response["messages"][-1].content)

Agent: I’m sorry, but I don’t have the Wi‑Fi password information available. If there’s anything else I can help you with—such as our menu, ordering details, or bakery policies—please let me know!


In [101]:
order_memory_config = {
    "configurable": {
        "thread_id": "signoff_order_memory"
    }
}

# First message
response = bakery_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I am Ravi. I want to order 2 plum cakes for December 24, eggless."
            }
        ]
    },
    config=order_memory_config
)

print("Agent:", response["messages"][-1].content)

Agent: Thank you, Ravi! Your order for **2 egg‑less plum cakes** on **December 24, 2026** has been placed successfully.  

If you need to make any changes or have any other requests, just let me know. Have a wonderful day!


In [102]:
response = bakery_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Actually, make it 3 cakes instead."}
        ]
    },
    config=order_memory_config
)

print("Agent:", response["messages"][-1].content)

Agent: Sure! Just to confirm, you’d like **3 egg‑less plum cakes** for **December 24, 2026** (instead of the 2 you previously requested). Should I go ahead and update your order?


## Task 15: Test Results and Evaluation

The Sweet Crumbs Home Bakery AI agent is evaluated using multiple test cases covering price lookup, knowledge retrieval, customer memory, order handling, and unsupported queries.

Each test is evaluated based on the correctness of the response, proper tool usage, retrieval accuracy, and prevention of hallucinations.

The overall pass rate is calculated to evaluate the reliability of the integrated agent.

In [103]:
test_results = {
    "Price lookup": True,
    "Payment policy": True,
    "Cake freshness": True,
    "Unsupported query": True,
    "Customer memory": True,
    "Memory isolation": True,
    "Order modification": True,
    "Opening hours": None,
    "Order logging": None
}

passed = sum(result is True for result in test_results.values())
failed = sum(result is False for result in test_results.values())
untested = sum(result is None for result in test_results.values())
total_tested = passed + failed

pass_rate = (passed / total_tested * 100) if total_tested else 0

print("TEST RESULTS")
print("-" * 35)

for test, result in test_results.items():
    status = "PASS" if result is True else "FAIL" if result is False else "NOT TESTED"
    print(f"{test}: {status}")

print("-" * 35)
print("Passed:", passed)
print("Failed:", failed)
print("Untested:", untested)
print(f"Pass rate: {pass_rate:.2f}%")

TEST RESULTS
-----------------------------------
Price lookup: PASS
Payment policy: PASS
Cake freshness: PASS
Unsupported query: PASS
Customer memory: PASS
Memory isolation: PASS
Order modification: PASS
Opening hours: NOT TESTED
Order logging: NOT TESTED
-----------------------------------
Passed: 7
Failed: 0
Untested: 2
Pass rate: 100.00%


### Evaluation

The integrated agent is evaluated using the test pass rate. Tests verify that the agent uses the appropriate tools, retrieves relevant information from ChromaDB, remembers conversation context, keeps customer memory isolated, and handles unsupported questions without fabricating answers.

A test is marked as passed only when the observed result matches the expected behavior. Untested scenarios are excluded from the pass-rate calculation.

The target pass rate is at least 80%.